## 04 - Recovering Depth 

## Imports

In [1]:
import numpy as np

## A Depth Problem

We previously covered how perspective projection, when leveraging camera intrinsics and extrinsics, can allow us to transform 3D points from the real world into 2D points on an image. 

Mapping from 3D to 2D represents an inherent problem with perspective projection: we lose the depth dimension (Z), which means that two objects that may be at different points along that Z axis will be mapped to the same $(x, y)$ coordinates. 

Let's illustrate this below. 

### Two Points - Different Z, Same (X, Y)


Recall our perspective projection equations 

$x = f\frac{X}{Z}$, $y = f\frac{Y}{Z}$

and suppose we have two points in the 3D camera coordinate plane $(X_1, Y_1, Z_1)$ and $(X_2, Y_2, Z_2)$ (we omit the $c$ subscript for brevity). 

It follows that these two points will map to the same $(x, y)$ if the following relationships hold true: 

$$ \frac{X_1}{Z_1} = \frac{X_2}{Z_2}, \frac{Y_1}{Z_1} = \frac{Y_2}{Z_2} $$




But how does this happen? Don't we have our $Z_c$ term from our intrinsic matrix? 

Well, we *did*, before we did our intrinsic multiplication. 

When we found the result of the intrinsic matrix multiplication 

$$
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
= 
\begin{bmatrix}
\frac{X_c}{Z_c} \\
\frac{Y_c}{Z_c}\\
1
\end{bmatrix}
$$

we lose our $Z_c$ value, and we cannot get this back. This is due to the property of a non-invertible transformation (again another matrix property you shouldn't worry about too much for now), so if we did try to reverse the intrinsic mulitplication from the previous notebook, we would get this equation 

$$
\begin{bmatrix}
X_c \\
Y_c \\
Z_c
\end{bmatrix}
= 
\lambda
K^{-1}
p
$$

where we are again left with our unknown $\lambda$ as our depth term, giving us infinitely many possible solutions. The $-1$ for our $K$ represents a matrix inverse. 

Fortunately, there are some solutions - let's go over them. 


### Recovering Depth: Homography with a Planar Constraint

Recall from our homography unit that the key assumption is that our 2D images must lie on the same 3D plane, such that we can get at least 4 points in each image that capture the same object. 

This assumption can help us recover depth, up to a point. Since our Z is constant given the above contrainst, we know the depth of objects - if they lie on this plane. We find this by performing some operations on our homography matrix $H$, known as a decomposition. 

If they don't lie on the same plane (and we cannot get those 4 points), then we are not able to recover depth. Therefore, we don't typically use this approach much in object detection scenarios. 

### Recovering Depth: Triangulation w/ 2 Cameras 

#### Single Camera View

Remember our 3D perspective projection 

$$p = \lambda K MP$$

As mentioned, we have this constraint of not being able to solve for $\lambda$. You can imagine that this situation looks like this: 

![](./ref_imgs/single_camera.png)

Again, as we've talked about, we have no way of finding the depth of the object because we just have a ray. However, we do know that we can take a ray, and if we add two other rays that intersect, we get a triangle. 

How do we create this triangle? 

#### Two Camera View - Triangulation

We can get this triangle by adding in a separate camera. By doing so, we get a single ray, just like our first one, pointing at our object. That forms the two legs of our triangle. The base is simply the distance between our two cameras! 

We can see this play out in the below diagram:

![](./ref_imgs/two_cameras.png)

This method is known as **triangulation**. It is quite a popular method, because of the fact that it is relatively simple to perform - simply add in a second camera! 

However, this method is not perfect. For one, we do have to know the distance between the two cameras. While this is not particularly difficult to guarantee, the more difficult problem is that if our cameras move at all (like during the robot driving), our lens gets distorted, any camera noise, or we have small intersection angles, our measurements can get thrown off tremendously.

#### Stereo Depth

**Stereo depth** is a specialized technique of triangulation that helps to determine depth. It comes from **stereo vision**, which is observation through two eyes (or in this case, cameras) and subsequent fusion of those images. Stereo depth calibrates cameras by aligning their intrinsics and extrinsics, which is usually done by rectifying photos after rotations/translations/other transformations to achieve alignment between the two cameras.

Then, we calculate the **disparity** between the images of each of the two cameras. 

We calculate disparity as $d = x_{left} - x_{right}$, which is the horizontal difference in pixels for corresponding points in our rectified images. 

Finally, we can calculate the depth as 

$$ Z = (f * B) / d $$

where $f$ is our focal length, $B$ is our baseline distance between the cameras, and $d$ is the disparity. 

This method is very useful, but does require us to perform the image rectification in the first place (which may be difficult), and also requires that our images are textured and in proper contrast and brightness (so that we can actually see pixel differences). 

#### Triangulation Over Time: Structure from Motion

As it turns out, we do not *need* to have a second camera to perform triangulation. We can actually perform triangulation with a single camera, provided that the camera is moving. 

Essentially, you capture the image at one point, move the camera, and then capture it at another point, and if you tracked the distance between the points where the camera took the shots, you have your triangle! 

This is admittedly an inferior form of triangulation, considering we have to have a trackable form of motion, and in a practical sense would have to make sure our robot stops and then is able to process the image. The sequential nature of the image capture also makes it inherently slower. 

### Recovering Depth: Perspective-n-Point

#### What does our Robot Know?

All of our discussions have been operating under a problematic assumption - that we actually know our extrinsics and intrinsics! 

Well, this isn't a problem for us humans, but our robot doesn't actually know any of these things. We have to tell our robot that it has Limelights attached to it, so it knows where they are - but how? 

#### Known Geometry

The triangulation methods we talked about don't make any assumptions about the objects we are detecting - but what if we have information on the kind of objects we're detecting, like AprilTags? 

With an AprilTag and its respective map, we are able to feed our robots some very key information. With the map, the robot knows where each of the AprilTags are located relative to the field. This represents the 3D real-world coordinates of each AprilTag. Additionally, Limelight has built in the 2D projections of AprilTags (the pixel coordinates of the 4 corners) using their fiducial tags. 

So, going back to our perspective projection equation 

$$ p = \lambda KMP $$

if we have our 2D point ($p$), our 3D point ($P$), and know the intrinsics of our camera ($K$), then the only unknowns we are left with is our depth scalar $\lambda$ and our extrinsic matrix $M$. 

The process of solving this is called **Perspective-n-Point**, or PnP. PnP attempts to estimate the camera extrinsics when given an object's known geometry. 



#### Solving for Unknowns

Recall our extrinsic multiplication 

$$ P_c = MP_w $$

which we can rewrite as 

$$ P_c = RP_w + t$$

Also recall our intrinsic multiplication resulting in 

$$
u = f_x \frac{X_c}{Z_c} + c_x
$$, 
$$
v = f_y \frac{Y_c}{Z_c} + c_y
$$

Due to the orthonormal nature of the rotation matrix $R$ (another matrix term you shouldn't worry about), we have 3 degrees of freedom, or 3 independent unknowns to solve for. Our translation matrix also has 3 degrees of freedom. Again, do not worry *how* we know this is the number of unknowns. 

We have 6 total unknowns, but only 2 constraints (our calculations for our 2D image coordinates $u$ and $v$). This means we need at least 3 points to calculate a solution to our $R$ and $t$, although only 3 points gives us ambiguous solutions, so we want 4 for a stable solution. 

These points must be non-collinear, or cannot all be in the same line. 

#### Constraining Depth

We know from above that our attempt to reverse the perspective projection equation yields 

$$
\begin{bmatrix}
X_c \\
Y_c \\
Z_c
\end{bmatrix}
= 
\lambda
K^{-1}
p
$$

which has infinitely many solutions thanks to our unknown $\lambda$.

To make the notation consistent, let's rewrite the left hand side with the $P_c$ term and write it for each individual point in an image as  

$$
P_i^c
= 
\lambda
K^{-1}
p_i
$$

Note that this occurs when we have a single point. When we have multiple points, we introduce the constraint from above that says 

$$ P_c = RP_w + t $$

or we can rewrite it for each individual point $i$ as 

$$ P_i^c = RP_i^w + t $$

and by setting those two equations equal to each other we get 

$$\lambda K^{-1} p_i = RP_i^w + t$$

and by multiplying both sides by $K$ we finally get 

$$Z_c p_i = K(RP_i^w + t)$$

We now have a solvable system that allows us to solve for depth (rewritten as $Z_c$ as a solvable parameter). There are various algorithms that solve this equation, such as EPnP or Levenberg-Marquadt. 

#### Shortcomings

Many of the algorithms/solvers aim to minimize projection error (difference of observed pixel and projected pixel) because ultimately PnP is an estimation and will make mistakes. 

Also, PnP works contingent on us having proper intrinsics on our camera, as well as not having noise in our detection (movement, poor exposure, etc.). 

### Recovering Depth: Monocular Depth Estimation

**Monocular Depth Estimation** is a powerful computer vision technique that predicts the distance information for every pixel from a single 2D image. It does so by using advanced ML techniques like CNNs and Transformer architecture to infer depth from cues like perspective or relative size based on similar images to the one being analyzed. 

Fortunately, there are models that do this for us, such as Depth Anything V2. 

Admittedly, while these techniques are exceptionally cool, like any ML-based technique, it is very prone to error based on the quality of the data. If we cannot provide images that are similar to what would be observed on the field, our depth estimation will be wildly inaccurate. 

Also, considering that many of these models are pre-trained, we would likely have to fine-tune these models on our own data, so they become adept at robotics tasks. This could be a fun thing for us to try...

## Big Takeaway

This notebook covered a lot of advanced perception techniques. The biggest takeaway here is not about the math formulas - it is about understanding the foundational concepts behind how we can detect objects. If you understand the need for extrinsics/intrinsics, and the problems with perspective projection, you will be able to leverage Limelights and other tools to solve many different tasks. 

For instance, Lmelights do much of the heavy lifting for PnP. So, we don't have to focus much on how to write the code, more so how to apply these concepts to our relevant tasks. 

Now, you have an understanding of how to find the locations of objects, also known as **pose estimation**. However, we have made one implicit assumption with all of these techniques - that our objects are stationary. 

How can we detect objects that move? 